# NB_hospital_daily_summary — Silver to Gold

Builds `adb_caresync.gold.hospital_daily_summary` (one row per hospital per day with lab activity) per the STTM `hospital_daily_summary` sheet: lab_results aggregated by hospital_id and test_date, LEFT JOIN hospitals and the calendar dimension, with a LAG() day-over-day trend. Full rebuild.

Notes: the calendar dimension is `adb_caresync.silver.calender` (project spelling). Silver `hospitals` and `lab_results` contain duplicate key rows, so both are de-duplicated on their business key (latest `update_timestamp`).

In [0]:
%sql
-- Gold: hospital_daily_summary | Grain: one row per hospital per day | Load: FULL rebuild
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary
COMMENT 'Daily lab activity per hospital with abnormal/critical rates and day-over-day test volume trend. One row per hospital per day.'
AS
WITH lab_results AS (
  SELECT * FROM adb_caresync.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
hospitals AS (
  SELECT hospital_id, hospital_name FROM adb_caresync.silver.hospitals
  QUALIFY row_number() OVER (PARTITION BY hospital_id ORDER BY update_timestamp DESC, landing_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                                   AS summary_date,
    COUNT(*)                                    AS total_tests,
    COUNT(DISTINCT patient_id)                  AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                   AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')        AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')        AS critical_test_count
  FROM lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    d.*,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(t.hospital_id AS INT)                                                   AS hospital_id,
  h.hospital_name                                                              AS hospital_name,
  t.summary_date                                                               AS summary_date,
  c.day_name                                                                   AS day_name,
  c.is_weekend                                                                 AS is_weekend,
  CAST(t.total_tests AS INT)                                                   AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                        AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                           AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                           AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                           AS critical_test_count,
  CAST(round(t.abnormal_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(round(t.critical_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                          AS prev_day_total_tests,
  CASE WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
       ELSE CAST(round((t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                                                          AS test_volume_change_pct
FROM trend t
LEFT JOIN hospitals h                   ON t.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calender c ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
%python
# Return the number of records written to gold.hospital_daily_summary (full rebuild = total rows in table)
# ADF: read via @activity('<NotebookActivity>').output.runOutput
target_table = "adb_caresync.gold.hospital_daily_summary"
record_count = spark.table(target_table).count()
print(f"FULL load completed for {target_table}. record_count = {record_count}")
dbutils.notebook.exit(str(record_count))